# Session 14 · Case study: tokens and semantic similarity

**Block 1 practice.** Tokenise reviews and compare token counts; compute semantic similarity between review sentences.

Needs `sentence-transformers` and `tiktoken` (see the session README). The first run downloads `all-MiniLM-L6-v2` (about 90 MB). No API key needed.

Theory: [01-from-words-to-vectors.md](../theory/01-from-words-to-vectors.md).

*Author: course team, licence CC-BY-4.0.*

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

# find the repository root (the folder that contains case-study/), wherever the notebook runs
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "case-study").exists())
DATA = ROOT / "case-study" / "data"
reviews = pd.read_parquet(DATA / "train_sample.parquet")
print(reviews.shape)

In [ ]:
import tiktoken
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
wordpiece = model.tokenizer                         # tokeniser of the embedding model (BERT type)
bpe = tiktoken.get_encoding("o200k_base")           # tokeniser of GPT-4o-type models
print("input limit of the embedding model:", model.max_seq_length, "tokens")

## 1. One review, three ways of splitting it

In [ ]:
text = reviews.loc[reviews["text"].str.len().between(80, 160), "text"].iloc[3]
print(text)
print("words     ", len(text.split()), text.split()[:12])
print("WordPiece ", len(wordpiece.tokenize(text)), wordpiece.tokenize(text)[:12])
print("BPE       ", len(bpe.encode(text)), [bpe.decode([i]) for i in bpe.encode(text)][:12])

## 2. Token counts of 2,000 reviews

In [ ]:
sample = reviews.groupby("label").sample(n=667, random_state=0).iloc[:2000]
docs = sample["title"] + ". " + sample["text"]
counts = pd.DataFrame({
    "label": sample["label"],
    "words": docs.str.split().str.len(),
    "wordpiece": docs.map(lambda t: len(wordpiece.tokenize(t))),
    "bpe": docs.map(lambda t: len(bpe.encode(t))),
})
print("tokens per word:", (counts[["wordpiece", "bpe"]].sum() / counts["words"].sum()).round(2).to_dict())
print("share longer than the model limit:", round((counts["wordpiece"] > model.max_seq_length).mean(), 3))
counts.groupby("label")[["words", "wordpiece", "bpe"]].median()

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(6, 4))
for label, part in counts.groupby("label"):
    ax.hist(part["wordpiece"].clip(upper=400), bins=40, alpha=0.5, label=label)
ax.axvline(model.max_seq_length, color="black", ls="--", label="model limit (256)")
ax.set_xlabel("WordPiece tokens per review (clipped at 400)")
ax.set_ylabel("reviews")
ax.legend()
plt.show()

**Question.** Which class writes the longest reviews? What happens to the end of a review that is longer than 256 tokens when it is embedded?

## 3. Semantic similarity between review sentences

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

sentences = [
    "The batteries died after two days.",
    "It stopped working almost immediately.",
    "Broke within a week, very disappointed.",
    "Excellent value, works perfectly.",
    "Does exactly what it promises, great buy.",
    "Smells great and lasts all day.",
    "The smell is awful and it gave me a headache.",
    "Arrived quickly and well packaged.",
]
E = model.encode(sentences, normalize_embeddings=True)
emb_sim = pd.DataFrame(E @ E.T, index=range(len(sentences)), columns=range(len(sentences)))
tfidf_sim = pd.DataFrame(cosine_similarity(TfidfVectorizer().fit_transform(sentences)))
for i, s in enumerate(sentences):
    print(i, s)
emb_sim.round(2).style.background_gradient(cmap="Blues", vmin=0, vmax=1)

In [ ]:
tfidf_sim.round(2).style.background_gradient(cmap="Blues", vmin=0, vmax=1)

## 4. Where do embeddings and TF-IDF disagree most?

In [ ]:
pairs = []
for i in range(len(sentences)):
    for j in range(i + 1, len(sentences)):
        pairs.append({"i": i, "j": j, "embedding": emb_sim.iloc[i, j], "tfidf": tfidf_sim.iloc[i, j]})
pairs = pd.DataFrame(pairs)
pairs["difference"] = pairs["embedding"] - pairs["tfidf"]
for _, p in pairs.sort_values("difference", ascending=False).head(3).iterrows():
    print(f"{p.embedding:.2f} vs {p.tfidf:.2f}: {sentences[int(p.i)]} | {sentences[int(p.j)]}")
for _, p in pairs.sort_values("difference").head(2).iterrows():
    print(f"{p.embedding:.2f} vs {p.tfidf:.2f}: {sentences[int(p.i)]} | {sentences[int(p.j)]}")

## Your turn

1. Sentences 5 and 6 are about the same topic with opposite opinions. How similar are they according to the embeddings? What does this mean for sentiment classification with embeddings?
2. Add three sentences from real reviews (for example from `reviews["text"]`), including one German or Spanish review if you find one (`reviews["text"].str.contains(" muy ")`), and look at their similarities.
3. Count the tokens of a 300-word German text with `bpe` and compare with an English translation. What does this mean for API costs?

In [ ]:
# your code here